# Annotation Union Merge + Disagreement Analysis

This notebook merges Reva and Ryan multi-label annotations:

1. Parse labels with blank/NaN → `_unknown`
2. Take the union of label sets
3. Remove `_unknown` when substantive labels exist
4. Serialize output labels deterministically

It then enriches `merged_glossary.tsv` with `source_domain` and `tier` columns from the provenance-classified glossary, and runs a disagreement analysis on both labeling tasks to assess whether the naive union-merge scheme is appropriate.

Writes:
- `merged_labels.tsv`
- `merged_glossary.tsv` (with `source_domain`, `tier`)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Resolve annotations directory whether notebook is run from annotations/ or repo root.
CWD = Path.cwd()
ANNOTATIONS_DIR = CWD if (CWD / "reva_labels.tsv").exists() else CWD / "annotations"

if not (ANNOTATIONS_DIR / "reva_labels.tsv").exists():
    raise FileNotFoundError(f"Could not find annotation files in {ANNOTATIONS_DIR}")

ANNOTATIONS_DIR

In [ ]:
def parse_labels(value) -> frozenset:
    """Parse comma-separated label string to frozenset of stripped labels.
    NaN, empty string, and whitespace-only all return frozenset({'_unknown'})."""
    if pd.isna(value) or str(value).strip() == "":
        return frozenset({'_unknown'})
    return frozenset(lbl.strip() for lbl in str(value).split(",") if lbl.strip())


def clean_merged(label_set: frozenset) -> frozenset:
    """Remove _unknown from sets that also contain substantive labels.
    Keep _unknown only when it is the sole label."""
    real = label_set - {'_unknown'}
    return real if real else frozenset({'_unknown'})


def serialize(label_set: frozenset) -> str:
    """Sort labels alphabetically and join with ', '."""
    return ", ".join(sorted(label_set))


def explode_counts(series_of_sets: pd.Series) -> pd.Series:
    """Count label frequency across items (multi-label explode)."""
    exploded = pd.Series([label for s in series_of_sets for label in s])
    return exploded.value_counts().sort_values(ascending=False)


def merge_task(
    reva_path: Path,
    ryan_path: Path,
    join_keys: list[str],
    label_col: str,
    merged_col: str,
    out_path: Path,
):
    reva = pd.read_csv(reva_path, sep="\t", dtype=str)
    ryan = pd.read_csv(ryan_path, sep="\t", dtype=str)

    merged = reva.merge(
        ryan[[*join_keys, label_col]],
        on=join_keys,
        how="inner",
        suffixes=("_reva_orig", "_ryan_orig"),
    )

    # Reconstruct the reva schema column name because merge suffixes rename overlaps.
    merged[label_col] = merged[f"{label_col}_reva_orig"]

    # Preserve original strings in explicit columns.
    merged[f"{label_col}_reva"] = merged[f"{label_col}_reva_orig"]
    merged[f"{label_col}_ryan"] = merged[f"{label_col}_ryan_orig"]

    reva_sets = merged[f"{label_col}_reva"].apply(parse_labels)
    ryan_sets = merged[f"{label_col}_ryan"].apply(parse_labels)

    merged_raw = [a | b for a, b in zip(reva_sets, ryan_sets)]
    merged_clean = [clean_merged(s) for s in merged_raw]

    merged[merged_col] = [serialize(s) for s in merged_clean]

    # Output file: all columns from reva file + required merge columns.
    base_cols = list(reva.columns)
    out_cols = [*base_cols, f"{label_col}_reva", f"{label_col}_ryan", merged_col]
    merged_out = merged[out_cols].copy()
    merged_out.to_csv(out_path, sep="\t", index=False)

    both_unknown = sum(
        (a == frozenset({'_unknown'}) and b == frozenset({'_unknown'}))
        for a, b in zip(reva_sets, ryan_sets)
    )
    unknown_stripped = sum(
        ('_unknown' in raw and cleaned != raw)
        for raw, cleaned in zip(merged_raw, merged_clean)
    )
    unique_labels = len(set().union(*merged_clean)) if merged_clean else 0

    stats = {
        'total_items': int(len(merged_out)),
        'both_unknown': int(both_unknown),
        'unknown_stripped': int(unknown_stripped),
        'unique_labels': int(unique_labels),
    }

    counts = pd.DataFrame({
        'Reva': explode_counts(reva_sets),
        'Ryan': explode_counts(ryan_sets),
        'Merged': explode_counts(pd.Series(merged_clean)),
    }).fillna(0).astype(int).sort_values('Merged', ascending=False)

    diagnostics = pd.DataFrame([
        {'category': 'both_unknown', 'count': int(both_unknown)},
        {'category': 'unknown_stripped', 'count': int(unknown_stripped)},
        {
            'category': 'unknown_retained_only',
            'count': int(sum(s == frozenset({'_unknown'}) for s in merged_clean)),
        },
    ])

    return merged_out, stats, counts, diagnostics

In [ ]:
labels_merged, labels_stats, labels_counts, labels_diag = merge_task(
    reva_path=ANNOTATIONS_DIR / "reva_labels.tsv",
    ryan_path=ANNOTATIONS_DIR / "ryan_labels.tsv",
    join_keys=["dataset", "raw_target_label"],
    label_col="dest_label",
    merged_col="dest_label_merged",
    out_path=ANNOTATIONS_DIR / "merged_labels.tsv",
)

glossary_merged, glossary_stats, glossary_counts, glossary_diag = merge_task(
    reva_path=ANNOTATIONS_DIR / "reva_glossary.tsv",
    ryan_path=ANNOTATIONS_DIR / "ryan_glossary.tsv",
    join_keys=["term"],
    label_col="inferred_target",
    merged_col="inferred_target_merged",
    out_path=ANNOTATIONS_DIR / "merged_glossary.tsv",
)

print("Wrote:")
print(ANNOTATIONS_DIR / "merged_labels.tsv")
print(ANNOTATIONS_DIR / "merged_glossary.tsv")

In [ ]:
# Join source_domain and tier from the provenance-classified glossary into merged_glossary.tsv.
# glossary.tsv is written by 00_glossary_formatter.ipynb and already carries these columns.
_provenance = pd.read_csv(
    ANNOTATIONS_DIR.parent / 'outputs' / 'glossary' / 'glossary.tsv',
    sep='\t',
    usecols=['term', 'source_domain', 'tier'],
)
glossary_merged = glossary_merged.merge(_provenance, on='term', how='left')
glossary_merged.to_csv(ANNOTATIONS_DIR / 'merged_glossary.tsv', sep='\t', index=False)

tier_coverage = glossary_merged['tier'].notna().sum()
print(f'Re-wrote merged_glossary.tsv with tier ({tier_coverage}/{len(glossary_merged)} rows matched)')
glossary_merged[['term', 'source_domain', 'tier', 'inferred_target_merged']].head(8)

In [ ]:
print("Labels task")
print("-----------")
print(f"Total items:                 {labels_stats['total_items']}")
print(f"Both _unknown after merge:   {labels_stats['both_unknown']}")
print(f"_unknown stripped from union: {labels_stats['unknown_stripped']}")
print(f"Unique labels in merged:     {labels_stats['unique_labels']}")
print()
print("Glossary task")
print("-------------")
print(f"Total items:                 {glossary_stats['total_items']}")
print(f"Both _unknown after merge:   {glossary_stats['both_unknown']}")
print(f"_unknown stripped from union: {glossary_stats['unknown_stripped']}")
print(f"Unique labels in merged:     {glossary_stats['unique_labels']}")

In [ ]:
def plot_top_label_breakdown(counts_df: pd.DataFrame, title: str, top_n: int = 20) -> None:
    top = counts_df.head(top_n).copy()

    ax = top[["Reva", "Ryan", "Merged"]].plot(
        kind="bar",
        figsize=(14, 6),
        width=0.85,
    )
    ax.set_title(title)
    ax.set_xlabel("Label")
    ax.set_ylabel("Count across items")
    ax.legend(loc="upper right")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def plot_unknown_diagnostics(diag_df: pd.DataFrame, title: str) -> None:
    ax = diag_df.plot(kind="bar", x="category", y="count", legend=False, figsize=(8, 4), color="#5a88c8")
    ax.set_title(title)
    ax.set_xlabel("Diagnostic category")
    ax.set_ylabel("Count")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

## Visual Breakdown: Labels Task

In [ ]:
plot_top_label_breakdown(labels_counts, "Labels Task: Top Labels by Merged Count", top_n=20)
plot_unknown_diagnostics(labels_diag, "Labels Task: _unknown Handling Diagnostics")
labels_counts.head(20)

## Visual Breakdown: Glossary Task

In [ ]:
plot_top_label_breakdown(glossary_counts, "Glossary Task: Top Labels by Merged Count", top_n=20)
plot_unknown_diagnostics(glossary_diag, "Glossary Task: _unknown Handling Diagnostics")
glossary_counts.head(20)

---

## Disagreement Analysis

The union-merge strategy resolves disagreements by taking the union of label sets. This is maximally inclusive but may be inappropriate when annotators assign *incompatible* labels rather than complementary ones.

Two questions motivate re-examining the naive scheme:

1. **Are most disagreements additive?**  
   An *additive* disagreement means one annotator's real-label set is a (strict) subset of the other's — the union just adds the extra labels without any conflict.  One annotator is empty (assigned `_unknown` or nothing) while the other has labels also counts as additive.

2. **How many cases are completely non-overlapping (disjoint)?**  
   A *disjoint* disagreement means the two annotators share zero labels — the union silently merges two incompatible judgements, with no signal about which (if either) is correct.

Categories:
| Class | Meaning |
|---|---|
| `exact_agreement` | Both annotators assigned identical real labels |
| `both_unknown` | Neither assigned any real labels |
| `additive` | Real labels differ; one set ⊆ the other (union is benign) |
| `partial_overlap` | Real labels differ; non-empty intersection; neither is a subset |
| `disjoint` | Real labels differ; no shared labels at all |

In [ ]:
_ORDER = ['exact_agreement', 'both_unknown', 'additive', 'partial_overlap', 'disjoint']
_COLORS = {
    'exact_agreement': '#2ca02c',
    'both_unknown':    '#aec7e8',
    'additive':        '#1f77b4',
    'partial_overlap': '#ff7f0e',
    'disjoint':        '#d62728',
}
_DISPLAY = {
    'exact_agreement': 'Exact agreement',
    'both_unknown':    'Both unknown',
    'additive':        'Additive (one ⊆ other)',
    'partial_overlap': 'Partial overlap',
    'disjoint':        'Disjoint (no shared labels)',
}


def classify_disagreement(reva_val, ryan_val) -> str:
    """
    Classify the relationship between two annotators' label sets.

    Strips _unknown placeholders before testing subset/overlap relationships
    so that 'I don't know' never counts as a substantive label assignment.
    """
    r = parse_labels(reva_val) - {'_unknown'}
    y = parse_labels(ryan_val) - {'_unknown'}

    if not r and not y:
        return 'both_unknown'
    if r == y:
        return 'exact_agreement'
    # One annotator left no real labels — the union is benign
    if not r or not y:
        return 'additive'
    ix = r & y
    if not ix:
        return 'disjoint'
    if r <= y or y <= r:
        return 'additive'
    return 'partial_overlap'


def disagreement_summary(df: pd.DataFrame, reva_col: str, ryan_col: str):
    classes = df.apply(
        lambda row: classify_disagreement(row[reva_col], row[ryan_col]), axis=1
    )
    counts = classes.value_counts().reindex(_ORDER, fill_value=0)
    total = len(df)
    disagree_n = int(total - counts['exact_agreement'] - counts['both_unknown'])
    summary = pd.DataFrame({
        'count': counts,
        'pct_total': (counts / total * 100).round(1),
    })
    return summary, disagree_n, classes


labels_dis_summary, labels_dis_n, _   = disagreement_summary(
    labels_merged, 'dest_label_reva', 'dest_label_ryan'
)
glossary_dis_summary, glossary_dis_n, _ = disagreement_summary(
    glossary_merged, 'inferred_target_reva', 'inferred_target_ryan'
)

In [ ]:
def _print_disagreement_report(summary: pd.DataFrame, task_name: str, disagree_n: int) -> None:
    total = summary['count'].sum()
    print(f"{task_name}  (n={total})")
    print("-" * 56)
    for cls in _ORDER:
        cnt = summary.loc[cls, 'count']
        pct = summary.loc[cls, 'pct_total']
        print(f"  {_DISPLAY[cls]:<30s}: {cnt:>4d}  ({pct:>5.1f}%)")
    print()
    if disagree_n > 0:
        print(f"  Among {disagree_n} true disagreements (excl. exact_agreement + both_unknown):")
        for cls in ['additive', 'partial_overlap', 'disjoint']:
            cnt = summary.loc[cls, 'count']
            pct = cnt / disagree_n * 100
            print(f"    {_DISPLAY[cls]:<30s}: {cnt:>4d}  ({pct:>5.1f}%)")
    print()


_print_disagreement_report(labels_dis_summary, "Labels task", labels_dis_n)
_print_disagreement_report(glossary_dis_summary, "Glossary task", glossary_dis_n)

In [ ]:
# Overview: all five categories side-by-side for both tasks
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, summary, title in [
    (axes[0], labels_dis_summary,  f'Labels task  (n={labels_dis_summary["count"].sum()})'),
    (axes[1], glossary_dis_summary, f'Glossary task  (n={glossary_dis_summary["count"].sum()})'),
]:
    disp_labels = [_DISPLAY[c] for c in _ORDER]
    counts_vals = [summary.loc[c, 'count'] for c in _ORDER]
    colors_vals = [_COLORS[c] for c in _ORDER]
    bars = ax.barh(disp_labels, counts_vals, color=colors_vals)
    for bar, val in zip(bars, counts_vals):
        if val > 0:
            ax.text(
                bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                str(val), va='center', fontsize=9,
            )
    ax.set_title(title)
    ax.set_xlabel('Number of items')
    ax.invert_yaxis()
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Full agreement-class breakdown by task', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Zoom in: structure of true disagreements only (excludes exact_agreement + both_unknown)
_DIS_CATS = ['additive', 'partial_overlap', 'disjoint']

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, summary, disagree_n, title in [
    (axes[0], labels_dis_summary,   labels_dis_n,
     f'Labels task\n({labels_dis_n} true disagreements)'),
    (axes[1], glossary_dis_summary, glossary_dis_n,
     f'Glossary task\n({glossary_dis_n} true disagreements)'),
]:
    if disagree_n == 0:
        ax.text(0.5, 0.5, 'No disagreements', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        ax.axis('off')
        continue

    vals = [summary.loc[c, 'count'] for c in _DIS_CATS]
    colors_sub = [_COLORS[c] for c in _DIS_CATS]
    display_sub = [_DISPLAY[c] for c in _DIS_CATS]

    wedges, texts, autotexts = ax.pie(
        vals,
        labels=display_sub,
        colors=colors_sub,
        autopct='%1.1f%%',
        startangle=90,
        pctdistance=0.72,
        wedgeprops={'linewidth': 0.6, 'edgecolor': 'white'},
    )
    for t in autotexts:
        t.set_fontsize(9)
    ax.set_title(title)

plt.suptitle('Structure of disagreements\n(exact_agreement and both_unknown excluded)', fontsize=11)
plt.tight_layout()
plt.show()

### Disagreement case tables

Full listing of every item where Reva and Ryan disagreed, with its disagreement class.  
`both_unknown` and `exact_agreement` rows are excluded — only the 201 (labels) and 97 (glossary) true disagreements are shown.

Sorted by class so the most problematic cases (`disjoint`) are grouped together.

In [ ]:
_DIS_ORDER = {'disjoint': 0, 'partial_overlap': 1, 'additive': 2}

# --- Labels task ---
labels_merged['disagreement_class'] = labels_merged.apply(
    lambda row: classify_disagreement(row['dest_label_reva'], row['dest_label_ryan']), axis=1
)
labels_disagreements = (
    labels_merged[labels_merged['disagreement_class'].isin(_DIS_ORDER)]
    .assign(_sort=lambda df: df['disagreement_class'].map(_DIS_ORDER))
    .sort_values(['_sort', 'dataset', 'raw_target_label'])
    .drop(columns=['_sort'])
    [['dataset', 'raw_target_label', 'dest_label_reva', 'dest_label_ryan', 'disagreement_class']]
    .reset_index(drop=True)
)

print(f"Labels task — {len(labels_disagreements)} true disagreements "
      f"({labels_disagreements['disagreement_class'].value_counts().to_dict()})")
display(labels_disagreements)

In [ ]:
# --- Glossary task ---
glossary_merged['disagreement_class'] = glossary_merged.apply(
    lambda row: classify_disagreement(row['inferred_target_reva'], row['inferred_target_ryan']), axis=1
)
glossary_disagreements = (
    glossary_merged[glossary_merged['disagreement_class'].isin(_DIS_ORDER)]
    .assign(_sort=lambda df: df['disagreement_class'].map(_DIS_ORDER))
    .sort_values(['_sort', 'term'])
    .drop(columns=['_sort'])
    [['term', 'inferred_target_reva', 'inferred_target_ryan', 'tier', 'disagreement_class']]
    .reset_index(drop=True)
)

print(f"Glossary task — {len(glossary_disagreements)} true disagreements "
      f"({glossary_disagreements['disagreement_class'].value_counts().to_dict()})")
display(glossary_disagreements)

---

## Disagreement Resolution

The earlier analysis identified several systematic disagreement patterns that the
naive union-merge handles poorly. This section replaces `inferred_target_merged`
and `dest_label_merged` with rule-based resolutions and writes
`merged_glossary_resolved.tsv` / `merged_labels_resolved.tsv`.

**Resolution path (applied in order):**

| Path | Condition | Action |
|---|---|---|
| `exact` | Identical real labels | Keep as-is |
| `both_unknown` | Both assigned `_unknown`/NaN | Keep `_unknown` |
| `rule1_unknown_vs_specific` | One side is purely `_unknown`, other has real labels | Use real labels |
| `additive_superset` | One real-label set ⊆ other (both have real labels) | Union (superset) |
| `rule2_transgender_granularity` | Only disagreement is `gender_transgender_women` vs `gender_transgender_unspecified` | Use `_unspecified`; `_men` for `biological man` |
| `rule3_self_referential_vs_target` | One side has `self_referential_white_supremacist`, other has target labels | Persona-dependent sub-rules |
| `rule4_specificity_explosion` | One side has ≥ 4 `race_` labels, other has only `unknown_minority` | Use `unknown_minority` |
| `partial_superset` | Partial overlap (not subset) | Union (superset) |
| `unresolved_disjoint` | Disjoint, no rule matched | Union + flag for manual review |

`_unknown` is treated as the empty set for subset/intersection checks throughout.

In [ ]:
"""
Resolution helpers and four named rule functions.

_unknown is the sentinel for 'annotator could not assign a label'.
It is stripped before every subset/intersection check so that
  reva=_unknown, ryan=race_black  →  additive  (not disjoint)
  reva=unknown_minority, ryan=race_black  →  disjoint  (unknown_minority is a real label)
"""
import math as _math

# ---------------------------------------------------------------------------
# Label constants
# ---------------------------------------------------------------------------
_SELF_REF    = 'self_referential_white_supremacist'
_TRANS_WOMEN = 'gender_transgender women'
_TRANS_UNSP  = 'gender_transgender unspecified'
_TRANS_MEN   = 'gender_transgender men'
_UNKNOWN     = '_unknown'
_UNK_MIN     = 'unknown_minority'


# ---------------------------------------------------------------------------
# Parsing / serialization
# ---------------------------------------------------------------------------

def parse_raw_labels(value) -> frozenset:
    """Parse a comma-separated label string to frozenset.

    NaN, None, and empty strings map to frozenset({'_unknown'}).
    Keeps '_unknown' as an explicit member so callers can test for it.
    """
    if value is None or (isinstance(value, float) and _math.isnan(value)):
        return frozenset({_UNKNOWN})
    s = str(value).strip()
    if not s:
        return frozenset({_UNKNOWN})
    return frozenset(lbl.strip() for lbl in s.split(',') if lbl.strip())


def effective_labels(raw: frozenset) -> frozenset:
    """Strip _unknown placeholders; used for all subset/intersection tests."""
    return raw - {_UNKNOWN}


def serialize_labels(labels: frozenset) -> str:
    """Deterministic comma-separated serialization, alphabetically sorted."""
    real = labels - {_UNKNOWN}
    if not real:
        return _UNKNOWN
    return ', '.join(sorted(real))


# ---------------------------------------------------------------------------
# Classification
# ---------------------------------------------------------------------------

def classify_pair(reva_raw, ryan_raw) -> str:
    """Classify the relationship between two annotators' label sets.

    _unknown is treated as the empty set for subset/intersection checks.

    Returns one of:
        'exact'        – identical real labels (both may also have _unknown)
        'both_unknown' – neither annotator assigned any real labels
        'additive'     – one real-label set is a (possibly strict) subset of
                         the other, including the case where one side is empty
                         after stripping _unknown
        'partial'      – non-empty intersection; neither is a subset
        'disjoint'     – real labels share no elements
    """
    r = effective_labels(parse_raw_labels(reva_raw))
    y = effective_labels(parse_raw_labels(ryan_raw))

    if not r and not y:
        return 'both_unknown'
    if r == y:
        return 'exact'
    if not r or not y:
        return 'additive'  # one side empty → subset of any non-empty set
    ix = r & y
    if not ix:
        return 'disjoint'
    if r <= y or y <= r:
        return 'additive'
    return 'partial'


# ---------------------------------------------------------------------------
# Rule 1 – Unknown vs. specific
# ---------------------------------------------------------------------------

def rule1_unknown_vs_specific(r_raw: frozenset, y_raw: frozenset) -> frozenset | None:
    """One annotator assigned _unknown/NaN; the other assigned real labels.

    Applies to ADDITIVE cases where one side's raw value is purely the
    _unknown sentinel.  Resolution: trust the annotator who produced
    specific labels — the one who wrote _unknown simply had no information.

    Examples (labels task): 'minorites' (_unknown → unknown_minority),
    'nazi' (_unknown → politics_alt right), 'social justice warriors'
    (_unknown → politics_leftist).

    Examples (glossary task): '23/16', 'Sonnenrad', 'based' where Reva
    left _unknown and Ryan supplied self_referential_white_supremacist.

    Returns the specific label set, or None if the condition is not met.
    """
    r_unk_only = r_raw <= {_UNKNOWN}  # True if raw set ⊆ {_unknown}
    y_unk_only = y_raw <= {_UNKNOWN}

    r_eff = effective_labels(r_raw)
    y_eff = effective_labels(y_raw)

    if r_unk_only and y_eff:
        return y_eff
    if y_unk_only and r_eff:
        return r_eff
    return None


# ---------------------------------------------------------------------------
# Rule 2 – Transgender granularity
# ---------------------------------------------------------------------------

def rule2_transgender_granularity(r: frozenset, y: frozenset) -> frozenset | None:
    """The ONLY source of disagreement is among the three transgender labels.

    Condition: when all non-transgender labels are stripped, r == y; and the
    transgender labels differ only between {_women, _unspecified, _men}.

    Resolution:
    - If gender_transgender_men appears in either set, use _men.
      (Term targets trans men specifically — e.g., 'biological man'.)
    - Otherwise use gender_transgender_unspecified.
      Rationale: TERF-coded terms like 'biological woman', 'uterus-haver',
      '#WomenWontWheesht', 'AFAB trans woman' target trans people broadly;
      collapsing to _women adds false precision and omits non-binary targets.

    Returns the resolved frozenset, or None if the condition is not met.
    """
    _CONTESTED = {_TRANS_WOMEN, _TRANS_UNSP, _TRANS_MEN}

    r_base = r - _CONTESTED
    y_base = y - _CONTESTED

    # Rule only fires when ALL disagreement is within the contested trans labels
    if r_base != y_base:
        return None

    r_trans = r & _CONTESTED
    y_trans = y & _CONTESTED
    all_trans = r_trans | y_trans

    if not all_trans:
        return None  # No transgender labels involved at all

    # Special case: trans men explicitly named by at least one annotator
    if _TRANS_MEN in all_trans:
        return r_base | {_TRANS_MEN}

    # Standard case: women vs unspecified → resolve to unspecified
    if _TRANS_WOMEN in all_trans and _TRANS_UNSP in all_trans:
        return r_base | {_TRANS_UNSP}

    return None  # Both annotators agreed on the same trans label (shouldn't be disjoint)


# ---------------------------------------------------------------------------
# Rule 3 – Self-referential vs. target framing
# ---------------------------------------------------------------------------

def rule3_self_referential_vs_target(
    r: frozenset, y: frozenset, persona: str = ''
) -> tuple[frozenset, str] | None:
    """One annotator uses self_referential_white_supremacist; other uses target labels.

    Applies when exactly one annotator's effective labels include
    self_referential_white_supremacist and the sets are disjoint overall.

    Sub-rules keyed on persona_in_group:

    white supremacist persona
        (a) Non-self_ref side contains ONLY unknown_minority (or nothing after
            stripping) → term is primarily an in-group signal with no clear
            external target.  Use self_referential_white_supremacist.
            Examples: Amerimutt.

        (b) Non-self_ref side has ≥ 3 specific labels (not unknown_minority)
            → term is BOTH self-referential AND targets external groups.
            Use the union of both sets; set merge_flag='self_ref_and_target'.
            Example: RWDS (targets race_black, race_middle eastern,
            religion_jewish, religion_muslim, politics_communist).

        (c) Non-self_ref side has 1–2 specific labels → term targets an
            external group; drop self_referential.
            Examples: Reagan → politics_conservative.

    other personas (racist, anti-liberal, etc.)
        self_referential_white_supremacist is a mis-assignment for these
        terms.  Use the other annotator's labels regardless.
        Examples: White Lives Matter → race_black; security from unrest →
        unknown_minority; election integrity → unknown_minority.

    Returns (resolved_frozenset, merge_flag) or None if rule does not apply.
    merge_flag is 'self_ref_and_target' for RWDS-type cases, else ''.
    """
    r_has_sr = _SELF_REF in r
    y_has_sr = _SELF_REF in y

    # Rule only fires when exactly one side carries self_ref
    if r_has_sr == y_has_sr:
        return None

    target_labels = y if r_has_sr else r  # The non-self_ref side
    persona_lower = str(persona).lower().strip()
    is_ws_persona = 'white supremacist' in persona_lower

    if is_ws_persona:
        specific = target_labels - {_UNK_MIN}  # Exclude unknown_minority
        if len(specific) >= 3:
            # RWDS-type: keep both framings
            return (r | y, 'self_ref_and_target')
        elif not specific:
            # Target side is empty or only unknown_minority → in-group signal
            return (frozenset({_SELF_REF}), '')
        else:
            # 1–2 specific targets (e.g., politics_conservative) → external framing
            return (target_labels, '')
    else:
        # Non-WS persona: self_ref is inappropriate
        if target_labels:
            return (target_labels, '')
        return None


# ---------------------------------------------------------------------------
# Rule 4 – Specificity explosion
# ---------------------------------------------------------------------------

def rule4_specificity_explosion(
    r: frozenset, y: frozenset, threshold: int = 4
) -> frozenset | None:
    """One annotator exploded a pan-group term into ≥ threshold race_ labels;
    the other used unknown_minority (and nothing else).

    Resolution: use unknown_minority.

    Rationale: pan-group terms like 'colored people', 'non-whites',
    'brown people' target non-white people broadly.  Decomposing them into
    6+ specific race labels assigns false precision and will artificially
    inflate per-group counts in downstream disparity analyses.

    Threshold is 4 by default (catches explosions of 4–7 race labels while
    leaving single-label or two-label specificity differences unaffected).

    The rule is applied symmetrically — it fires regardless of which
    annotator produced the explosion.

    Returns frozenset({'unknown_minority'}), or None if condition not met.
    """
    r_race = {lbl for lbl in r if lbl.startswith('race_')}
    y_race = {lbl for lbl in y if lbl.startswith('race_')}

    r_only_unk_min = r == {_UNK_MIN}
    y_only_unk_min = y == {_UNK_MIN}

    if len(r_race) >= threshold and y_only_unk_min:
        return frozenset({_UNK_MIN})
    if len(y_race) >= threshold and r_only_unk_min:
        return frozenset({_UNK_MIN})
    return None


# ---------------------------------------------------------------------------
# Main resolution function
# ---------------------------------------------------------------------------

def resolve_pair(reva_raw, ryan_raw, persona: str = '') -> dict:
    """Apply the four named rules to a single annotator pair.

    Parameters
    ----------
    reva_raw, ryan_raw : raw label strings (may be NaN)
    persona : value of persona_in_group column (glossary task only; pass ''
              for the labels task)

    Returns a dict with:
        merged_labels : frozenset of resolved labels
        merge_method  : one of the method name strings
        merge_flag    : '' normally; 'unresolved_disjoint' or
                        'self_ref_and_target' for special cases
    """
    r_raw = parse_raw_labels(reva_raw)
    y_raw = parse_raw_labels(ryan_raw)
    r = effective_labels(r_raw)
    y = effective_labels(y_raw)
    cls = classify_pair(reva_raw, ryan_raw)

    # ── Exact ──────────────────────────────────────────────────────────────
    if cls == 'exact':
        resolved = r if r else frozenset({_UNKNOWN})
        return {'merged_labels': resolved, 'merge_method': 'exact', 'merge_flag': ''}

    # ── Both unknown ────────────────────────────────────────────────────────
    if cls == 'both_unknown':
        return {'merged_labels': frozenset({_UNKNOWN}),
                'merge_method': 'both_unknown', 'merge_flag': ''}

    # ── Additive ────────────────────────────────────────────────────────────
    if cls == 'additive':
        # Rule 1: one side is purely _unknown → trust the other annotator
        r1 = rule1_unknown_vs_specific(r_raw, y_raw)
        if r1 is not None:
            return {'merged_labels': r1,
                    'merge_method': 'rule1_unknown_vs_specific', 'merge_flag': ''}
        # Genuine additive (both had real labels, one is a subset)
        return {'merged_labels': r | y,
                'merge_method': 'additive_superset', 'merge_flag': ''}

    # ── Partial overlap ─────────────────────────────────────────────────────
    if cls == 'partial':
        return {'merged_labels': r | y,
                'merge_method': 'partial_superset', 'merge_flag': ''}

    # ── Disjoint: apply rules 2 → 3 → 4 → fallback ─────────────────────────
    # Rule 2: transgender label granularity
    r2 = rule2_transgender_granularity(r, y)
    if r2 is not None:
        return {'merged_labels': r2,
                'merge_method': 'rule2_transgender_granularity', 'merge_flag': ''}

    # Rule 3: self-referential vs. target framing
    r3 = rule3_self_referential_vs_target(r, y, persona)
    if r3 is not None:
        resolved, flag = r3
        return {'merged_labels': resolved,
                'merge_method': 'rule3_self_referential_vs_target',
                'merge_flag': flag}

    # Rule 4: specificity explosion
    r4 = rule4_specificity_explosion(r, y)
    if r4 is not None:
        return {'merged_labels': r4,
                'merge_method': 'rule4_specificity_explosion', 'merge_flag': ''}

    # Fallback: unresolved disjoint — use superset and flag for manual review
    return {'merged_labels': r | y,
            'merge_method': 'unresolved_disjoint',
            'merge_flag': 'unresolved_disjoint'}

In [ ]:
# Self-contained: reads merged TSVs, applies resolution, writes outputs.
# Requires: ANNOTATIONS_DIR (defined in cell 2), rule functions (defined above).

_glossary_raw = pd.read_csv(ANNOTATIONS_DIR / 'merged_glossary.tsv', sep='\t', dtype=str)
_labels_raw   = pd.read_csv(ANNOTATIONS_DIR / 'merged_labels.tsv',   sep='\t', dtype=str)

def _apply_resolution(
    df: pd.DataFrame,
    reva_col: str,
    ryan_col: str,
    merged_col: str,
    persona_col: str | None = None,
) -> pd.DataFrame:
    """Vectorised application of resolve_pair over every row of df."""
    out = df.copy()
    results = [
        resolve_pair(
            row[reva_col],
            row[ryan_col],
            str(row[persona_col]) if persona_col and pd.notna(row.get(persona_col)) else '',
        )
        for _, row in out.iterrows()
    ]
    out[merged_col]    = [serialize_labels(res['merged_labels']) for res in results]
    out['merge_method'] = [res['merge_method'] for res in results]
    out['merge_flag']   = [res['merge_flag']   for res in results]
    return out

glossary_resolved = _apply_resolution(
    _glossary_raw,
    reva_col='inferred_target_reva',
    ryan_col='inferred_target_ryan',
    merged_col='inferred_target_merged',
    persona_col='persona_in_group',
)

labels_resolved = _apply_resolution(
    _labels_raw,
    reva_col='dest_label_reva',
    ryan_col='dest_label_ryan',
    merged_col='dest_label_merged',
    persona_col=None,
)

glossary_resolved.to_csv(ANNOTATIONS_DIR / 'merged_glossary_resolved.tsv', sep='\t', index=False)
labels_resolved.to_csv(  ANNOTATIONS_DIR / 'merged_labels_resolved.tsv',   sep='\t', index=False)
print(f'Wrote merged_glossary_resolved.tsv  ({len(glossary_resolved)} rows)')
print(f'Wrote merged_labels_resolved.tsv    ({len(labels_resolved)} rows)')

# ── Summary ──────────────────────────────────────────────────────────────────

_METHOD_ORDER = [
    'exact',
    'both_unknown',
    'rule1_unknown_vs_specific',
    'additive_superset',
    'rule2_transgender_granularity',
    'rule3_self_referential_vs_target',
    'rule4_specificity_explosion',
    'partial_superset',
    'unresolved_disjoint',
]

def _print_summary(df: pd.DataFrame, task: str) -> None:
    counts = df['merge_method'].value_counts()
    total  = len(df)
    print(f'\n=== {task} RESOLUTION SUMMARY ===')
    for m in _METHOD_ORDER:
        n    = counts.get(m, 0)
        note = '  ← needs manual review' if m == 'unresolved_disjoint' else ''
        print(f'  {m:<40s}: {n:>4d}{note}')
    print(f'  {"TOTAL":<40s}: {total:>4d}')

_print_summary(glossary_resolved, 'GLOSSARY TASK')
_print_summary(labels_resolved,   'LABELS TASK')

# ── Unresolved disjoint cases ─────────────────────────────────────────────────

print('\n=== UNRESOLVED DISJOINT CASES (need manual review) ===')

_g_unres = glossary_resolved[glossary_resolved['merge_flag'] == 'unresolved_disjoint']
print(f'\nGLOSSARY ({len(_g_unres)} cases):')
for _, row in _g_unres.iterrows():
    print(f"  {row['term']!r:50}  reva={row['inferred_target_reva']!r}  ryan={row['inferred_target_ryan']!r}")

_l_unres = labels_resolved[labels_resolved['merge_flag'] == 'unresolved_disjoint']
print(f'\nLABELS ({len(_l_unres)} cases):')
for _, row in _l_unres.iterrows():
    print(f"  [{row['dataset']}] {row['raw_target_label']!r:35}  reva={row['dest_label_reva']!r}  ryan={row['dest_label_ryan']!r}")

# ── self_ref_and_target flag ──────────────────────────────────────────────────

_sr_and_t = glossary_resolved[glossary_resolved['merge_flag'] == 'self_ref_and_target']
if not _sr_and_t.empty:
    print(f'\nGLOSSARY — self_ref_and_target ({len(_sr_and_t)} case(s)):')
    for _, row in _sr_and_t.iterrows():
        print(f"  {row['term']!r:30}  merged={row['inferred_target_merged']!r}")

In [ ]:
# ---------------------------------------------------------------------------
# Unit tests — one known example per rule from the actual annotation data
# ---------------------------------------------------------------------------
import traceback as _tb

_PASS, _FAIL = 0, 0

def _check(name: str, got, expected) -> None:
    global _PASS, _FAIL
    if got == expected:
        print(f'  PASS  {name}')
        _PASS += 1
    else:
        print(f'  FAIL  {name}')
        print(f'        got      = {got!r}')
        print(f'        expected = {expected!r}')
        _FAIL += 1

def _res(rv, ry, persona=''):
    d = resolve_pair(rv, ry, persona)
    return serialize_labels(d['merged_labels']), d['merge_method'], d['merge_flag']

print('── Classification ─────────────────────────────────────────────────────')
_check('classify: exact',        classify_pair('race_black', 'race_black'), 'exact')
_check('classify: both_unknown', classify_pair('_unknown', float('nan')),   'both_unknown')
_check('classify: additive (one unk)', classify_pair('_unknown', 'race_black'), 'additive')
_check('classify: additive (subset)', classify_pair('race_black', 'race_black, religion_jewish'), 'additive')
_check('classify: partial',      classify_pair('race_black, gender_men', 'race_black, gender_women'), 'partial')
_check('classify: disjoint',     classify_pair('race_black', 'race_white'), 'disjoint')
_check('classify: unk_min NOT unk', classify_pair('unknown_minority', 'race_black'), 'disjoint')

print()
print('── Rule 1: unknown vs. specific ───────────────────────────────────────')
# Labels task: 'minorites' → reva=_unknown, ryan=unknown_minority
merged, method, flag = _res('_unknown', 'unknown_minority')
_check('rule1 | labels: minorites', (merged, method), ('unknown_minority', 'rule1_unknown_vs_specific'))

# Labels task: 'nazi' → reva=_unknown, ryan=politics_alt right
merged, method, flag = _res('_unknown', 'politics_alt right')
_check('rule1 | labels: nazi', (merged, method), ('politics_alt right', 'rule1_unknown_vs_specific'))

# Glossary task: '23/16' → reva=_unknown, ryan=self_referential_white_supremacist
merged, method, flag = _res('_unknown', 'self_referential_white_supremacist')
_check('rule1 | glossary: 23/16', (merged, method),
       ('self_referential_white_supremacist', 'rule1_unknown_vs_specific'))

# Reversed: ryan=_unknown
merged, method, flag = _res('race_black', float('nan'))
_check('rule1 | ryan=NaN', (merged, method), ('race_black', 'rule1_unknown_vs_specific'))

print()
print('── Rule 2: transgender granularity ────────────────────────────────────')
# Glossary: 'autogynephile' → reva=_unspecified, ryan=_women → use _unspecified
merged, method, flag = _res('gender_transgender unspecified', 'gender_transgender women')
_check('rule2 | unspecified vs women → unspecified',
       (merged, method), ('gender_transgender unspecified', 'rule2_transgender_granularity'))

# Glossary: '#WomenWontWheesht' → reva=_women, ryan=_unspecified → use _unspecified
merged, method, flag = _res('gender_transgender women', 'gender_transgender unspecified')
_check('rule2 | women vs unspecified → unspecified',
       (merged, method), ('gender_transgender unspecified', 'rule2_transgender_granularity'))

# Glossary: 'biological man' → reva=_men, ryan=_women → use _men (special case)
merged, method, flag = _res('gender_transgender men', 'gender_transgender women')
_check('rule2 | men vs women → men (biological man)',
       (merged, method), ('gender_transgender men', 'rule2_transgender_granularity'))

print()
print('── Rule 3: self-referential vs. target ────────────────────────────────')
# Glossary: 'White Lives Matter' (persona=racist) → use race_black
merged, method, flag = _res(
    'self_referential_white_supremacist', 'race_black', persona='racist')
_check('rule3 | non-WS persona → use target (White Lives Matter)',
       (merged, method), ('race_black', 'rule3_self_referential_vs_target'))

# Glossary: 'security from unrest' (persona=racist) → use unknown_minority
merged, method, flag = _res(
    'self_referential_white_supremacist', 'unknown_minority', persona='racist')
_check('rule3 | non-WS persona, unk_min → use unk_min (security from unrest)',
       (merged, method), ('unknown_minority', 'rule3_self_referential_vs_target'))

# Glossary: 'Amerimutt' (persona=white supremacist, target=unknown_minority) → use self_ref
merged, method, flag = _res(
    'self_referential_white_supremacist', 'unknown_minority', persona='white supremacist')
_check('rule3 | WS + unk_min target → self_ref (Amerimutt)',
       (merged, method), ('self_referential_white_supremacist', 'rule3_self_referential_vs_target'))

# Glossary: 'Reagan' (persona=white supremacist, 1 specific target) → use politics_conservative
merged, method, flag = _res(
    'self_referential_white_supremacist', 'politics_conservative', persona='white supremacist')
_check('rule3 | WS + 1 specific target → target (Reagan)',
       (merged, method), ('politics_conservative', 'rule3_self_referential_vs_target'))

# Glossary: 'RWDS' (persona=white supremacist, 5 specific targets) → union + flag
merged, method, flag = _res(
    'self_referential_white_supremacist',
    'race_black, race_middle eastern, religion_jewish, religion_muslim, politics_communist',
    persona='white supremacist')
_check('rule3 | WS + 5 targets → union (RWDS)',
       method, 'rule3_self_referential_vs_target')
_check('rule3 | RWDS flag', flag, 'self_ref_and_target')
_check('rule3 | RWDS includes self_ref',
       'self_referential_white_supremacist' in merged, True)

print()
print('── Rule 4: specificity explosion ──────────────────────────────────────')
# Labels: 'colored people' → reva=7 race labels, ryan=unknown_minority
_reva_explosion = ('race_asian, race_black, race_latinx, race_middle eastern, '
                   'race_native american, race_pacific islander, race_other')
merged, method, flag = _res(_reva_explosion, 'unknown_minority')
_check('rule4 | reva explosion (colored people)',
       (merged, method), ('unknown_minority', 'rule4_specificity_explosion'))

# Labels: 'brown people' → reva=unknown_minority, ryan=5 race labels (reversed)
_ryan_explosion = ('race_black, race_latinx, race_middle eastern, '
                   'race_native american, race_pacific islander')
merged, method, flag = _res('unknown_minority', _ryan_explosion)
_check('rule4 | ryan explosion (brown people)',
       (merged, method), ('unknown_minority', 'rule4_specificity_explosion'))

# Negative: 3 race labels should NOT trigger rule 4
merged, method, flag = _res('race_black, race_white, race_asian', 'unknown_minority')
_check('rule4 | 3 race labels → NOT rule4 (fallback expected)',
       method, 'unresolved_disjoint')

print()
print('── Fallback: unresolved disjoint ──────────────────────────────────────')
# 'NPC' (anti-liberal): politics_liberal vs politics_leftist
merged, method, flag = _res('politics_liberal', 'politics_leftist')
_check('fallback | NPC: liberal vs leftist',
       (method, flag), ('unresolved_disjoint', 'unresolved_disjoint'))
_check('fallback | NPC: superset retained',
       set(merged.split(', ')), {'politics_liberal', 'politics_leftist'})

print()
print(f'{"─"*60}')
print(f'Results: {_PASS} passed, {_FAIL} failed')